# 04 BM25

## Objetivos

* Implementar BM25 y comparar su ranking con el de TF-IDF + coseno sobre la misma colección de documentos.
* Calcular el aporte de cada término en BM25, identificar qué controlan $k_1$ y $b$, y justificar un cambio de posición usando las estadísticas de los documentos.

**1. Carga el corpus de Project Gutenberg**

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Recupere una tabla con los campos **Identificador de documento | Título | Texto**

Calcule la columna `tokens` para la tabla, con lista de tokens normalizados de cada documento.

Calcula DF e IDF.

In [ ]:
import os
import math
import string
from collections import Counter

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.datasets import load_files

# Ruta al corpus de Project Gutenberg (reutilizando la carpeta de la práctica anterior)
ruta_corpus = '../Modelo vectorial/txt-files'
corpus = load_files(ruta_corpus, encoding='utf-8', shuffle=False)

textos_originales = corpus.data
N = len(textos_originales)
print(f"Se cargaron {N} documentos.")

# Identificador de cada documento: nombre del archivo sin extensión (ej. libro_11)
ids_documentos = [
    os.path.basename(ruta).replace('.txt', '')
    for ruta in corpus.filenames
]

# Línea que empieza con "Title:" en la cabecera de Gutenberg
def obtener_titulo(texto):
    for linea in texto.split('\n'):
        if linea.startswith('Title:'):
            return linea.replace('Title:', '').strip()
    return 'Sin título'

titulos = [obtener_titulo(texto) for texto in textos_originales]

# Stopwords y preprocesamiento de la práctica anterior
stopwords = {
    "the", "and", "to", "of", "a", "i", "in", "it",
    "that", "was", "he", "is", "for", "as", "on", "with"
}

def preprocesar(texto):
    # 1. Minúsculas
    texto_limpio = texto.lower()
    # 2. Quitar puntuación
    texto_limpio = texto_limpio.translate(str.maketrans('', '', string.punctuation))
    # 3. Dividir en lista de palabras (tokenizar)
    palabras = texto_limpio.split()
    # 4. Filtrar stopwords y conservar solo palabras alfabéticas
    palabras_filtradas = [p for p in palabras if p not in stopwords and p.isalpha()]
    return palabras_filtradas

# Tabla con Identificador de documento | Título | Texto y columna tokens
tabla_corpus = pd.DataFrame({
    'Identificador de documento': ids_documentos,
    'Título': titulos,
    'Texto': textos_originales
})
tabla_corpus['tokens'] = [preprocesar(texto) for texto in tabla_corpus['Texto']]

pd.set_option('display.max_colwidth', 60)
print("\nTabla del corpus (con columna tokens):")
display(tabla_corpus)

# Cálculo de DF e IDF para todo el vocabulario
conteo_df = Counter()
for tokens_doc in tabla_corpus['tokens']:
    conteo_df.update(set(tokens_doc))

vocabulario = sorted(conteo_df.keys())

# IDF(t) = ln(N / DF(t)) + 1 
def calcular_idf(df, total_docs):
    return math.log(total_docs / df) + 1

df_terminos = {t: conteo_df[t] for t in vocabulario}
idf_terminos = {t: calcular_idf(conteo_df[t], N) for t in vocabulario}

tabla_df_idf = pd.DataFrame({
    'Término': vocabulario,
    'DF': [df_terminos[t] for t in vocabulario],
    'IDF': [round(idf_terminos[t], 4) for t in vocabulario]
})

print(f"\nTamaño del vocabulario: {len(vocabulario)} términos")
print("Muestra de DF e IDF (términos de ejemplo):")
terminos_muestra = ['zenda', 'flatland', 'harpoon', 'tarzan', 'whale', 'alice', 'constitution', 'sea', 'time', 'life']
display(tabla_df_idf[tabla_df_idf['Término'].isin(terminos_muestra)].sort_values('DF').reset_index(drop=True))

Se cargaron 100 documentos.

Tabla del corpus (con columna tokens):

Tamaño del vocabulario: 125315 términos
Muestra de DF e IDF (términos de ejemplo):


,Identificador de documento,Título,Texto,tokens
0,libro_1,The Declaration of Independence of the United States of ...,The Project Gutenberg eBook of The Declaration of Indepe...,"[project, gutenberg, ebook, declaration, independence, u..."
1,libro_10,The King James Version of the Bible,The Project Gutenberg eBook of The King James Version of...,"[project, gutenberg, ebook, king, james, version, bible,..."
2,libro_100,The Complete Works of William Shakespeare,The Project Gutenberg eBook of The Complete Works of Wil...,"[project, gutenberg, ebook, complete, works, william, sh..."
3,libro_101,The Hacker Crackdown: Law and Disorder on the Electronic...,The Project Gutenberg eBook of The Hacker Crackdown: Law...,"[project, gutenberg, ebook, hacker, crackdown, law, diso..."
4,libro_11,Alice's Adventures in Wonderland,The Project Gutenberg eBook of Alice's Adventures in Won...,"[project, gutenberg, ebook, alices, adventures, wonderla..."
...,...,...,...,...
95,libro_95,The prisoner of Zenda,The Project Gutenberg eBook of The prisoner of Zenda\r\n...,"[project, gutenberg, ebook, prisoner, zenda, this, ebook..."
96,libro_96,The Monster Men,The Project Gutenberg eBook of The Monster Men\r\n \r...,"[project, gutenberg, ebook, monster, men, this, ebook, u..."
97,libro_97,Flatland: A Romance of Many Dimensions,The Project Gutenberg eBook of Flatland: A Romance of Ma...,"[project, gutenberg, ebook, flatland, romance, many, dim..."
98,libro_98,A Tale of Two Cities,The Project Gutenberg eBook of A Tale of Two Cities\r\n ...,"[project, gutenberg, ebook, tale, two, cities, this, ebo..."


,Término,DF,IDF
0,zenda,1,5.6052
1,flatland,2,4.9120
2,harpoon,3,4.5066
3,tarzan,5,3.9957
4,whale,11,3.2073
5,alice,18,2.7148
6,constitution,39,1.9416
7,sea,69,1.3711
8,time,94,1.0619
9,life,100,1.0000


**2. Benchmarking**

Seleccione **ocho consultas en el idioma del corpus** y registra en una línea qué busca el usuario con cada una. Incluye términos muy frecuentes, términos poco frecuentes, y consultas de distintas longitudes. Usa $df$ para justificar las elecciones.

Para **TF-IDF + coseno** conserva la representación dispersa de la matriz.

Para **BM25** define $k_1=1.2$, $b=0.75$. Mantén intactas las estadísticas de la colección al cambiar de consulta.

Para cada consulta, ordena por puntuación descendente y desempata por `doc_id` ascendente (trata los IDs como cadenas en ambos métodos). Presenta hasta diez documentos con score positivo por método. Una consulta sin términos en el vocabulario produce dos listas vacías.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Definición de las 8 consultas en el idioma del corpus
consultas = [
    {
        'consulta': 'zenda',
        'necesidad': 'Buscar la novela de aventuras sobre el prisionero de Zenda.',
        'justificacion_df': 'Consulta de 1 término extremadamente raro (df=1: solo aparece en 1 documento).'
    },
    {
        'consulta': 'flatland dimensions',
        'necesidad': 'Encontrar la obra sobre un mundo geométrico de múltiples dimensiones.',
        'justificacion_df': 'Consulta corta (2 términos) que combina un término muy raro (flatland, df=2) con uno de frecuencia media-baja (dimensions, df=24).'
    },
    {
        'consulta': 'tarzan jungle apes',
        'necesidad': 'Historias de aventuras de Tarzán criado por los simios en la selva.',
        'justificacion_df': 'Consulta de 3 términos poco frecuentes (tarzan df=5, apes df=14, jungle df=19).'
    },
    {
        'consulta': 'alice rabbit queen king',
        'necesidad': 'Relatos de Alicia en el País de las Maravillas, el conejo blanco y los reyes.',
        'justificacion_df': 'Consulta de 4 términos que mezcla términos poco frecuentes (alice df=18, rabbit df=19) con frecuentes (queen df=49, king df=62).'
    },
    {
        'consulta': 'internet network email computer users',
        'necesidad': 'Guías introductorias sobre Internet, redes de computadoras y correo electrónico.',
        'justificacion_df': 'Consulta de 5 términos con términos específicos (internet df=12, users df=16) y términos muy frecuentes presentes en las cabeceras del corpus (df=100).'
    },
    {
        'consulta': 'constitution congress president rights government',
        'necesidad': 'Documentos históricos sobre la Constitución, el Congreso y el gobierno de Estados Unidos.',
        'justificacion_df': 'Consulta de 5 términos de frecuencia media y alta (congress df=33, constitution df=39, president df=41, rights df=59, government df=61).'
    },
    {
        'consulta': 'whale hunting ship captain harpoon sea',
        'necesidad': 'Novelas de aventuras sobre la caza de ballenas con arpón en altamar.',
        'justificacion_df': 'Consulta larga (6 términos) que combina términos muy raros (harpoon df=3, whale df=11) con términos frecuentes (captain df=47, ship df=51, hunting df=52, sea df=69).'
    },
    {
        'consulta': 'man life day time world great',
        'necesidad': 'Obras literarias o ensayos con reflexiones generales sobre el ser humano, la vida y el tiempo.',
        'justificacion_df': 'Consulta larga (6 términos) compuesta exclusivamente por términos muy frecuentes en la colección (df entre 84 y 100).'
    }
]

# Resumen de las 8 consultas y sus valores de DF
filas_consultas = []
for idx_q, c in enumerate(consultas, start=1):
    c['tokens'] = preprocesar(c['consulta'])
    dfs_consulta = {t: df_terminos.get(t, 0) for t in c['tokens']}
    filas_consultas.append({
        'N°': idx_q,
        'Consulta': c['consulta'],
        'Longitud': len(c['tokens']),
        'DF por término': ', '.join(f"{t} (df={d})" for t, d in dfs_consulta.items()),
        'Qué busca el usuario': c['necesidad'],
        'Justificación con DF': c['justificacion_df']
    })

pd.set_option('display.max_colwidth', None)
print("Consultas seleccionadas y justificación mediante DF:")
display(pd.DataFrame(filas_consultas))

# 2. Representación dispersa para TF-IDF + coseno
def sin_cambios(tokens):
    return tokens

vectorizador_tfidf = TfidfVectorizer(
    analyzer=sin_cambios,
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=False,
    norm=None
)
# Se conserva la matriz dispersa CSR 
matriz_tfidf_dispersa = vectorizador_tfidf.fit_transform(tabla_corpus['tokens'])
print(f"Matriz TF-IDF dispersa: tipo={type(matriz_tfidf_dispersa).__name__}, forma={matriz_tfidf_dispersa.shape}, elementos no cero={matriz_tfidf_dispersa.nnz}")

# 3. Estadísticas de la colección para BM25
K1_BASE = 1.2
B_BASE = 0.75

longitudes_docs = [len(tokens) for tokens in tabla_corpus['tokens']]
avgdl = sum(longitudes_docs) / N
conteos_docs = [Counter(tokens) for tokens in tabla_corpus['tokens']]
print(f"Estadísticas fijas de BM25: N={N}, avgdl={avgdl:.2f} tokens, k1={K1_BASE}, b={B_BASE}")

# Funciones de búsqueda y ranking con desempate por doc_id ascendente
def buscar_tfidf_coseno(tokens_consulta, top_k=10):
    tokens_validos = [t for t in tokens_consulta if t in df_terminos]
    if not tokens_validos:
        return []
    vector_q = vectorizador_tfidf.transform([tokens_consulta])
    scores = cosine_similarity(vector_q, matriz_tfidf_dispersa)[0]
    resultados = [
        {
            'doc_id': str(ids_documentos[i]),
            'titulo': titulos[i],
            'score': float(scores[i]),
            'indice_doc': i
        }
        for i in range(N) if scores[i] > 0
    ]
    resultados.sort(key=lambda r: (-r['score'], str(r['doc_id'])))
    return resultados[:top_k]

def aporte_termino_bm25(termino, indice_doc, k1=K1_BASE, b=B_BASE):
    if termino not in idf_terminos:
        return 0.0
    tf = conteos_docs[indice_doc][termino]
    if tf == 0:
        return 0.0
    dl = longitudes_docs[indice_doc]
    idf = idf_terminos[termino]
    numerador = tf * (k1 + 1.0)
    denominador = tf + k1 * (1.0 - b + b * (dl / avgdl))
    return idf * (numerador / denominador)

def buscar_bm25(tokens_consulta, k1=K1_BASE, b=B_BASE, top_k=10):
    tokens_validos = [t for t in tokens_consulta if t in df_terminos]
    if not tokens_validos:
        return []
    resultados = []
    for i in range(N):
        score = sum(aporte_termino_bm25(t, i, k1=k1, b=b) for t in tokens_validos)
        if score > 0:
            resultados.append({
                'doc_id': str(ids_documentos[i]),
                'titulo': titulos[i],
                'score': float(score),
                'indice_doc': i
            })
    resultados.sort(key=lambda r: (-r['score'], str(r['doc_id'])))
    return resultados[:top_k]

# Ejecución de las 8 consultas en ambos métodos
for c in consultas:
    c['top_tfidf'] = buscar_tfidf_coseno(c['tokens'], top_k=10)
    c['top_bm25'] = buscar_bm25(c['tokens'], k1=K1_BASE, b=B_BASE, top_k=10)

# Consulta vacía sin términos en el vocabulario
consulta_vacia = preprocesar('quantumcomputing blockchainxyzw')
print(
    "\nVerificación de consulta sin términos en el vocabulario ('quantumcomputing blockchainxyzw'):",
    "TF-IDF ->", buscar_tfidf_coseno(consulta_vacia),
    "| BM25 ->", buscar_bm25(consulta_vacia)
)

Consultas seleccionadas y justificación mediante DF:
Matriz TF-IDF dispersa: tipo=csr_matrix, forma=(100, 125315), elementos no cero=674571
Estadísticas fijas de BM25: N=100, avgdl=67085.77 tokens, k1=1.2, b=0.75

Verificación de consulta sin términos en el vocabulario ('quantumcomputing blockchainxyzw'): TF-IDF -> [] | BM25 -> []


,N°,Consulta,Longitud,DF por término,Qué busca el usuario,Justificación con DF
0,1,zenda,1,zenda (df=1),Buscar la novela de aventuras sobre el prisionero de Zenda.,Consulta de 1 término extremadamente raro (df=1: solo aparece en 1 documento).
1,2,flatland dimensions,2,"flatland (df=2), dimensions (df=24)",Encontrar la obra sobre un mundo geométrico de múltiples dimensiones.,"Consulta corta (2 términos) que combina un término muy raro (flatland, df=2) con uno de frecuencia media-baja (dimensions, df=24)."
2,3,tarzan jungle apes,3,"tarzan (df=5), jungle (df=19), apes (df=14)",Historias de aventuras de Tarzán criado por los simios en la selva.,"Consulta de 3 términos poco frecuentes (tarzan df=5, apes df=14, jungle df=19)."
3,4,alice rabbit queen king,4,"alice (df=18), rabbit (df=19), queen (df=49), king (df=62)","Relatos de Alicia en el País de las Maravillas, el conejo blanco y los reyes.","Consulta de 4 términos que mezcla términos poco frecuentes (alice df=18, rabbit df=19) con frecuentes (queen df=49, king df=62)."
4,5,internet network email computer users,5,"internet (df=12), network (df=100), email (df=100), computer (df=100), users (df=16)","Guías introductorias sobre Internet, redes de computadoras y correo electrónico.","Consulta de 5 términos con términos específicos (internet df=12, users df=16) y términos muy frecuentes presentes en las cabeceras del corpus (df=100)."
5,6,constitution congress president rights government,5,"constitution (df=39), congress (df=33), president (df=41), rights (df=59), government (df=61)","Documentos históricos sobre la Constitución, el Congreso y el gobierno de Estados Unidos.","Consulta de 5 términos de frecuencia media y alta (congress df=33, constitution df=39, president df=41, rights df=59, government df=61)."
6,7,whale hunting ship captain harpoon sea,6,"whale (df=11), hunting (df=52), ship (df=51), captain (df=47), harpoon (df=3), sea (df=69)",Novelas de aventuras sobre la caza de ballenas con arpón en altamar.,"Consulta larga (6 términos) que combina términos muy raros (harpoon df=3, whale df=11) con términos frecuentes (captain df=47, ship df=51, hunting df=52, sea df=69)."
7,8,man life day time world great,6,"man (df=84), life (df=100), day (df=86), time (df=94), world (df=98), great (df=89)","Obras literarias o ensayos con reflexiones generales sobre el ser humano, la vida y el tiempo.",Consulta larga (6 términos) compuesta exclusivamente por términos muy frecuentes en la colección (df entre 84 y 100).



**3. Compara los rankings**

Muestra por consulta una tabla como esta. Usa los IDs para comparar.

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | … | … | … | … |

Calcula cuántos IDs comparten los dos top 10 con $|T_{\mathrm{TFIDF}}\cap T_{\mathrm{BM25}}|$. Reporta también el tamaño real de cada lista y el primer documento de cada método. Dos listas pueden tener muchos IDs en común y ordenarlos de forma distinta.

Elige una consulta y dos documentos que cambien de posición o cuya distancia en el ranking cambie. Registra $|d|$, $f(t,d)$, $df(t)$ y el aporte BM25 de cada término de consulta para ambos documentos. Relaciona esas cifras con la saturación, la longitud o la rareza del término. Usa los scores de cada método para explicar su propio ranking.

In [ ]:
# 1. Tablas comparativas por consulta y cálculo de intersección
resumen_comparacion = []

for idx_q, c in enumerate(consultas, start=1):
    lista_tfidf = c['top_tfidf']
    lista_bm25 = c['top_bm25']
    max_filas = max(len(lista_tfidf), len(lista_bm25))

    ids_tfidf = {r['doc_id'] for r in lista_tfidf}
    ids_bm25 = {r['doc_id'] for r in lista_bm25}
    interseccion = ids_tfidf & ids_bm25

    primero_tfidf = f"{lista_tfidf[0]['doc_id']} ({lista_tfidf[0]['titulo']})" if lista_tfidf else 'Ninguno'
    primero_bm25 = f"{lista_bm25[0]['doc_id']} ({lista_bm25[0]['titulo']})" if lista_bm25 else 'Ninguno'

    resumen_comparacion.append({
        'N°': idx_q,
        'Consulta': c['consulta'],
        '|T_TFIDF|': len(lista_tfidf),
        '|T_BM25|': len(lista_bm25),
        '|T_TFIDF ∩ T_BM25|': len(interseccion),
        'Primer documento TF-IDF': primero_tfidf,
        'Primer documento BM25': primero_bm25
    })

    filas_tabla = []
    for pos in range(max_filas):
        if pos < len(lista_tfidf):
            id_tit_tfidf = f"{lista_tfidf[pos]['doc_id']} - {lista_tfidf[pos]['titulo']}"
            score_cos = round(lista_tfidf[pos]['score'], 6)
        else:
            id_tit_tfidf = '—'
            score_cos = None

        if pos < len(lista_bm25):
            id_tit_bm25 = f"{lista_bm25[pos]['doc_id']} - {lista_bm25[pos]['titulo']}"
            score_bm25 = round(lista_bm25[pos]['score'], 4)
        else:
            id_tit_bm25 = '—'
            score_bm25 = None

        filas_tabla.append({
            'Posición': pos + 1,
            'ID y título TF-IDF': id_tit_tfidf,
            'Coseno': score_cos,
            'ID y título BM25': id_tit_bm25,
            'Score BM25': score_bm25
        })

    print(f"\nConsulta {idx_q}: '{c['consulta']}'")
    print(f"  Tamaño lista TF-IDF: {len(lista_tfidf)} | Tamaño lista BM25: {len(lista_bm25)} | |T_TFIDF ∩ T_BM25| = {len(interseccion)}")
    print(f"  Top-1 TF-IDF: {primero_tfidf} | Top-1 BM25: {primero_bm25}")
    display(pd.DataFrame(filas_tabla))

print("\nResumen general de coincidencia en el Top-10 por consulta:")
display(pd.DataFrame(resumen_comparacion))

# 2. Análisis detallado de cambio de posición entre dos documentos
# Elegimos la consulta 4 ('alice rabbit queen king') y los documentos libro_11 y libro_12
def analizar_cambio_posicion(consulta_str, docs_comparar):
    c_obj = next(c for c in consultas if c['consulta'] == consulta_str)
    tokens_q = c_obj['tokens']
    pos_tfidf = {r['doc_id']: i + 1 for i, r in enumerate(buscar_tfidf_coseno(tokens_q, top_k=N))}
    pos_bm25 = {r['doc_id']: i + 1 for i, r in enumerate(buscar_bm25(tokens_q, top_k=N))}
    scores_cos = cosine_similarity(vectorizador_tfidf.transform([tokens_q]), matriz_tfidf_dispersa)[0]

    print(f"\n=== Análisis de cambio de posición para la consulta: '{consulta_str}' ===")
    print(f"Longitud media de la colección (avgdl): {avgdl:.2f} tokens")

    filas_resumen_docs = []
    for doc_id in docs_comparar:
        idx = ids_documentos.index(doc_id)
        score_bm = sum(aporte_termino_bm25(t, idx) for t in tokens_q)
        filas_resumen_docs.append({
            'Documento': f"{doc_id} ({titulos[idx]})",
            '|d| (longitud)': longitudes_docs[idx],
            '|d| / avgdl': round(longitudes_docs[idx] / avgdl, 4),
            'Posición TF-IDF': pos_tfidf.get(doc_id, '-'),
            'Score Coseno': round(float(scores_cos[idx]), 6),
            'Posición BM25': pos_bm25.get(doc_id, '-'),
            'Score BM25': round(score_bm, 4)
        })
    display(pd.DataFrame(filas_resumen_docs))

    filas_terminos = []
    for t in tokens_q:
        fila = {
            'Término (t)': t,
            'df(t)': df_terminos[t],
            'IDF(t)': round(idf_terminos[t], 4)
        }
        for doc_id in docs_comparar:
            idx = ids_documentos.index(doc_id)
            tf = conteos_docs[idx][t]
            aporte_bm = aporte_termino_bm25(t, idx)
            peso_tfidf = tf * idf_terminos[t]
            fila[f'f(t, {doc_id})'] = tf
            fila[f'Peso TF-IDF ({doc_id})'] = round(peso_tfidf, 4)
            fila[f'Aporte BM25 ({doc_id})'] = round(aporte_bm, 4)
        filas_terminos.append(fila)

    display(pd.DataFrame(filas_terminos))

analizar_cambio_posicion('alice rabbit queen king', ['libro_11', 'libro_12'])
analizar_cambio_posicion('constitution congress president rights government', ['libro_5', 'libro_18'])


Consulta 1: 'zenda'
  Tamaño lista TF-IDF: 1 | Tamaño lista BM25: 1 | |T_TFIDF ∩ T_BM25| = 1
  Top-1 TF-IDF: libro_95 (The prisoner of Zenda) | Top-1 BM25: libro_95 (The prisoner of Zenda)

Consulta 2: 'flatland dimensions'
  Tamaño lista TF-IDF: 10 | Tamaño lista BM25: 10 | |T_TFIDF ∩ T_BM25| = 9
  Top-1 TF-IDF: libro_97 (Flatland: A Romance of Many Dimensions) | Top-1 BM25: libro_97 (Flatland: A Romance of Many Dimensions)

Consulta 3: 'tarzan jungle apes'
  Tamaño lista TF-IDF: 10 | Tamaño lista BM25: 10 | |T_TFIDF ∩ T_BM25| = 9
  Top-1 TF-IDF: libro_78 (Tarzan of the Apes) | Top-1 BM25: libro_78 (Tarzan of the Apes)

Consulta 4: 'alice rabbit queen king'
  Tamaño lista TF-IDF: 10 | Tamaño lista BM25: 10 | |T_TFIDF ∩ T_BM25| = 5
  Top-1 TF-IDF: libro_12 (Through the Looking-Glass) | Top-1 BM25: libro_11 (Alice's Adventures in Wonderland)

Consulta 5: 'internet network email computer users'
  Tamaño lista TF-IDF: 10 | Tamaño lista BM25: 10 | |T_TFIDF ∩ T_BM25| = 9
  Top-1 TF-IDF: li

,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,libro_95 - The prisoner of Zenda,0.129667,libro_95 - The prisoner of Zenda,12.19


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,libro_97 - Flatland: A Romance of Many Dimensions,0.285711,libro_97 - Flatland: A Romance of Many Dimensions,15.9999
1,2,libro_35 - The Time Machine,0.011019,libro_35 - The Time Machine,5.1146
2,3,"libro_83 - From the Earth to the moon; and, round the moon",0.003442,"libro_83 - From the Earth to the moon; and, round the moon",5.0011
3,4,libro_41 - The Legend of Sleepy Hollow,0.001754,libro_82 - Ivanhoe: A Romance,4.1393
4,5,libro_49 - Surfing the Internet: An Introduction,0.001609,libro_22 - Roget's Thesaurus,3.9481
5,6,libro_82 - Ivanhoe: A Romance,0.000945,libro_18 - The Federalist Papers,3.7721
6,7,libro_96 - The Monster Men,0.000683,libro_49 - Surfing the Internet: An Introduction,3.7609
7,8,libro_18 - The Federalist Papers,0.000638,libro_96 - The Monster Men,3.7427
8,9,"libro_23 - Narrative of the Life of Frederick Douglass, an American Slave",0.000627,libro_41 - The Legend of Sleepy Hollow,3.7103
9,10,libro_64 - The gods of Mars,0.000602,libro_64 - The gods of Mars,3.4681


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,libro_78 - Tarzan of the Apes,0.482618,libro_78 - Tarzan of the Apes,21.0766
1,2,libro_81 - The Return of Tarzan,0.477046,libro_85 - The Beasts of Tarzan,21.0594
2,3,libro_85 - The Beasts of Tarzan,0.407754,libro_92 - Tarzan and the Jewels of Opar,21.0181
3,4,libro_92 - Tarzan and the Jewels of Opar,0.333270,libro_81 - The Return of Tarzan,21.0077
4,5,libro_90 - The Son of Tarzan,0.094096,libro_90 - The Son of Tarzan,20.8118
5,6,libro_96 - The Monster Men,0.042664,libro_68 - The warlord of Mars,9.8026
6,7,libro_64 - The gods of Mars,0.010037,"libro_72 - Thuvia, maid of Mars",6.2461
7,8,"libro_72 - Thuvia, maid of Mars",0.009135,libro_64 - The gods of Mars,6.2330
8,9,libro_21 - Three hundred Aesop’s fables,0.006197,libro_21 - Three hundred Aesop’s fables,5.9858
9,10,libro_3 - John F. Kennedy's Inaugural Address,0.005109,libro_96 - The Monster Men,5.8073


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,libro_12 - Through the Looking-Glass,0.454305,libro_11 - Alice's Adventures in Wonderland,18.7026
1,2,libro_11 - Alice's Adventures in Wonderland,0.446802,libro_45 - Anne of Green Gables,13.4619
2,3,libro_95 - The prisoner of Zenda,0.047271,libro_12 - Through the Looking-Glass,12.9432
3,4,libro_100 - The Complete Works of William Shakespeare,0.046812,libro_100 - The Complete Works of William Shakespeare,12.8487
4,5,libro_86 - A Connecticut Yankee in King Arthur's Court,0.040462,libro_77 - The House of the Seven Gables,12.6461
5,6,libro_10 - The King James Version of the Bible,0.025145,libro_54 - The Marvelous Land of Oz,11.1289
6,7,"libro_30 - The Bible, King James Version, Complete",0.025112,libro_78 - Tarzan of the Apes,10.6113
7,8,libro_77 - The House of the Seven Gables,0.022619,libro_51 - Anne of the Island,10.0252
8,9,libro_54 - The Marvelous Land of Oz,0.016849,libro_44 - The Song of the Lark,9.7064
9,10,libro_76 - Adventures of Huckleberry Finn,0.016182,libro_98 - A Tale of Two Cities,9.6463


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,libro_75 - E-mail 101,0.366332,libro_80 - The Online World,19.5888
1,2,libro_49 - Surfing the Internet: An Introduction,0.352288,libro_40 - Northwestnet NUSIRG Internet Guide,19.5456
2,3,libro_39 - Hitchhiker's Guide to the Internet,0.336053,libro_34 - Zen and the Art of the Internet,19.5326
3,4,libro_34 - Zen and the Art of the Internet,0.299772,"libro_53 - Workshop on Electronic Texts: Proceedings, 9-10 June 1992",19.3661
4,5,libro_80 - The Online World,0.247115,libro_49 - Surfing the Internet: An Introduction,19.3358
5,6,libro_56 - NREN for All: Insurmountable Opportunity,0.207886,"libro_38 - The Jargon File, Version 2.9.10, 01 Jul 1992",19.2047
6,7,"libro_53 - Workshop on Electronic Texts: Proceedings, 9-10 June 1992",0.099494,libro_75 - E-mail 101,19.1786
7,8,libro_101 - The Hacker Crackdown: Law and Disorder on the Electronic Frontier,0.072476,libro_56 - NREN for All: Insurmountable Opportunity,18.9367
8,9,"libro_38 - The Jargon File, Version 2.9.10, 01 Jul 1992",0.052168,libro_39 - Hitchhiker's Guide to the Internet,18.9336
9,10,libro_79 - Terminal Compromise,0.027812,libro_101 - The Hacker Crackdown: Law and Disorder on the Electronic Frontier,18.9224


... (4 outputs left out)

,Documento,|d| (longitud),|d| / avgdl,Posición TF-IDF,Score Coseno,Posición BM25,Score BM25
0,libro_11 (Alice's Adventures in Wonderland),18918,0.2820,2,0.446802,1,18.7026
1,libro_12 (Through the Looking-Glass),20669,0.3081,1,0.454305,3,12.9432


,Término (t),df(t),IDF(t),"f(t, libro_11)",Peso TF-IDF (libro_11),Aporte BM25 (libro_11),"f(t, libro_12)",Peso TF-IDF (libro_12),Aporte BM25 (libro_12)
0,alice,18,2.7148,383,1039.7678,5.9639,440,1194.5113,5.9647
1,rabbit,19,2.6607,43,114.4114,5.7792,0,0.0000,0.0000
2,queen,49,1.7133,65,111.3677,3.7375,172,294.6962,3.7568
3,king,62,1.4780,60,88.6821,3.2219,62,91.6382,3.2217


,Documento,|d| (longitud),|d| / avgdl,Posición TF-IDF,Score Coseno,Posición BM25,Score BM25
0,libro_5 (The United States Constitution),5518,0.0823,1,0.174577,8,18.3871
1,libro_18 (The Federalist Papers),132513,1.9753,2,0.146709,1,19.5003


,Término (t),df(t),IDF(t),"f(t, libro_5)",Peso TF-IDF (libro_5),Aporte BM25 (libro_5),"f(t, libro_18)",Peso TF-IDF (libro_18),Aporte BM25 (libro_18)
0,constitution,39,1.9416,19,36.8906,4.1891,473,918.3808,4.2529
1,congress,33,2.1087,29,61.1512,4.5800,121,255.1482,4.5607
2,president,41,1.8916,32,60.5311,4.1134,122,230.7750,4.0918
3,rights,59,1.5276,1,1.5276,2.4459,152,232.2002,3.3155
4,government,61,1.4943,5,7.4715,3.0586,846,1264.1747,3.2794


### Explicación del cambio de posición con las estadísticas de los documentos

1. **Caso 1: Consulta `'alice rabbit queen king'` (`libro_11` vs `libro_12`)**
   * **Cambio en el ranking:** En **TF-IDF + coseno**, `libro_12` (*Through the Looking-Glass*, $\text{cos} = 0.454338$) ocupa la **posición 1** y `libro_11` (*Alice's Adventures in Wonderland*, $\text{cos} = 0.446809$) queda en la **posición 2**. En **BM25**, el orden se invierte: `libro_11` sube a la **posición 1** ($\text{score} = 18.7026$) y `libro_12` pasa a la **posición 3** ($\text{score} = 12.9432$).
   * **Saturación de frecuencia ($k_1$):** En TF-IDF el peso crece linealmente con $f(t,d)$. Como `libro_12` repite `alice` $440$ veces (frente a $383$ en `libro_11`) y `queen` $172$ veces (frente a $65$ en `libro_11`), sus pesos TF-IDF brutos ($1194.51$ y $294.69$) dominan el producto punto y le dan mayor similitud coseno. En cambio, en **BM25** ambas frecuencias ya están en la zona de **saturación** ($f(t,d) \gg k_1$), por lo que pasar de $383$ a $440$ apariciones de `alice` apenas cambia el aporte de $5.9639$ a $5.9647$, y pasar de $65$ a $172$ en `queen` solo lo mueve de $3.7375$ a $3.7568$.
   * **Rareza del término ($df(t)$) y cobertura:** El término `rabbit` es poco frecuente en la colección ($df = 19$, $\text{IDF} = 2.6607$). Aparece $43$ veces en `libro_11` (aportando $+5.7792$ puntos en BM25), pero **$0$ veces** en `libro_12` (aporte $0.0000$). Además, ambos documentos son cortos respecto a la media ($|d| = 18\,918$ y $20\,669$ frente a $\text{avgdl} = 67\,085.77$). Así, BM25 premia a `libro_11` por cubrir los **cuatro términos** (incluido el término raro `rabbit`) en lugar de premiar la sobre-repetición de `alice` y `queen` en `libro_12`.

2. **Caso 2: Consulta `'constitution congress president rights government'` (`libro_5` vs `libro_18`)**
   * **Cambio en el ranking:** En **TF-IDF + coseno**, `libro_5` (*The United States Constitution*, $|d| = 5\,518$) ocupa la **posición 1** ($\text{cos} = 0.174577$) y `libro_18` (*The Federalist Papers*, $|d| = 132\,513$) la **posición 2** ($\text{cos} = 0.146709$). En **BM25**, `libro_18` sube a la **posición 1** ($19.5003$) y `libro_5` baja a la **posición 8** ($18.3871$).
   * **Longitud ($|d|$) frente a saturación:** En similitud coseno, al dividir por la norma euclidiana, el documento muy corto `libro_5` ($|d|/\text{avgdl} = 0.0823$) resulta fuertemente favorecido porque casi todo su vocabulario gira en torno a esos términos. En **BM25**, aunque `libro_5` recibe bonificación por ser corto, sus frecuencias en `rights` ($f = 1$) y `government` ($f = 5$) aún no saturan la función (aportan $2.4459$ y $3.0586$), mientras que `libro_18` —pese a la penalización por longitud ($|d|/\text{avgdl} = 1.9753$)— tiene frecuencias tan altas en los cinco términos ($473, 121, 122, 152, 846$) que alcanza la saturación en todos ellos, superando a `libro_5` por $+1.1132$ puntos.

**4. Identifica el efecto de los parámetros**

Mantén fijos la colección y el preprocesamiento. Escoge una consulta con coincidencias en documentos de distintas longitudes y frecuencias.

| Ejecución | $k_1$ | $b$ | Qué examinar |
|---|---:|---:|---|
| Base | 1.2 | 0.75 | Posiciones y aportes de referencia |
| Sin ajuste por longitud | 1.2 | 0.00 | Cambios en documentos más cortos o más largos que la media |
| Mayor efecto de las repeticiones | 2.0 | 0.75 | Cambios en el aporte de términos con frecuencias distintas |

Muestra las posiciones de los mismos documentos en las tres configuraciones. Si no se mueven, compara sus aportes por término y explica por qué el cambio de puntuación no alteró su orden.

In [ ]:
# Escogemos la consulta 'alice rabbit queen king', que presenta coincidencias en documentos
# cortos (libro_11, libro_12), cercanos a la media (libro_45, libro_77) y muy largos (libro_100),
# así como frecuencias muy altas (cientos o miles de apariciones) y muy bajas (1 a 11 apariciones).
consulta_param = 'alice rabbit queen king'
tokens_param = preprocesar(consulta_param)

configuraciones = [
    ('Base', 1.2, 0.75),
    ('Sin ajuste por longitud', 1.2, 0.00),
    ('Mayor efecto de las repeticiones', 2.0, 0.75)
]

rankings_config = {}
posiciones_config = {}
scores_config = {}

for nombre_cfg, k1_val, b_val in configuraciones:
    ranking_completo = buscar_bm25(tokens_param, k1=k1_val, b=b_val, top_k=N)
    rankings_config[nombre_cfg] = ranking_completo[:10]
    posiciones_config[nombre_cfg] = {r['doc_id']: idx + 1 for idx, r in enumerate(ranking_completo)}
    scores_config[nombre_cfg] = {r['doc_id']: r['score'] for r in ranking_completo}

# Seguimiento de los mismos documentos (Top-6 de la configuración Base) en las tres configuraciones
docs_seguimiento = [r['doc_id'] for r in rankings_config['Base'][:6]]

filas_posiciones = []
for doc_id in docs_seguimiento:
    idx = ids_documentos.index(doc_id)
    dl = longitudes_docs[idx]
    filas_posiciones.append({
        'Documento': f"{doc_id} - {titulos[idx]}",
        '|d|': dl,
        '|d| / avgdl': round(dl / avgdl, 2),
        'Pos. Base (1.2, 0.75)': posiciones_config['Base'][doc_id],
        'Score Base': round(scores_config['Base'][doc_id], 4),
        'Pos. Sin ajuste (1.2, 0.00)': posiciones_config['Sin ajuste por longitud'][doc_id],
        'Score (b=0)': round(scores_config['Sin ajuste por longitud'][doc_id], 4),
        'Pos. Mayor rep. (2.0, 0.75)': posiciones_config['Mayor efecto de las repeticiones'][doc_id],
        'Score (k1=2)': round(scores_config['Mayor efecto de las repeticiones'][doc_id], 4)
    })

print(f"Consulta analizada: '{consulta_param}' (avgdl = {avgdl:.2f} tokens)")
print("Posiciones y puntuaciones de los mismos documentos en las tres configuraciones:")
display(pd.DataFrame(filas_posiciones))

# Aportes por término para los documentos analizados en las 3 configuraciones
filas_aportes = []
for doc_id in docs_seguimiento[:5]:
    idx = ids_documentos.index(doc_id)
    dl = longitudes_docs[idx]
    for nombre_cfg, k1_val, b_val in configuraciones:
        fila = {
            'Documento': f"{doc_id} ({titulos[idx][:28]})",
            '|d|/avgdl': round(dl / avgdl, 2),
            'Ejecución': f"{nombre_cfg} (k1={k1_val}, b={b_val})",
            'Posición': posiciones_config[nombre_cfg][doc_id],
            'Score Total': round(scores_config[nombre_cfg][doc_id], 4)
        }
        for t in tokens_param:
            tf = conteos_docs[idx][t]
            ap = aporte_termino_bm25(t, idx, k1=k1_val, b=b_val)
            fila[f"{t} (f={tf})"] = f"{ap:.4f} (tf={tf})"
        filas_aportes.append(fila)

print("\nComparación detallada de los aportes por término en cada configuración:")
display(pd.DataFrame(filas_aportes))

Consulta analizada: 'alice rabbit queen king' (avgdl = 67085.77 tokens)
Posiciones y puntuaciones de los mismos documentos en las tres configuraciones:

Comparación detallada de los aportes por término en cada configuración:


,Documento,|d|,|d| / avgdl,"Pos. Base (1.2, 0.75)",Score Base,"Pos. Sin ajuste (1.2, 0.00)",Score (b=0),"Pos. Mayor rep. (2.0, 0.75)",Score (k1=2)
0,libro_11 - Alice's Adventures in Wonderland,18918,0.28,1,18.7026,1,18.5376,1,25.3743
1,libro_45 - Anne of Green Gables,71743,1.07,2,13.4619,3,13.6459,4,15.5977
2,libro_12 - Through the Looking-Glass,20669,0.31,3,12.9432,4,12.8895,2,17.6044
3,libro_100 - The Complete Works of William Shakespeare,720393,10.74,4,12.8487,2,16.9465,3,15.9933
4,libro_77 - The House of the Seven Gables,73531,1.10,5,12.6461,5,12.8556,5,15.1491
5,libro_54 - The Marvelous Land of Oz,29070,0.43,6,11.1289,9,10.2696,6,14.1047


,Documento,|d|/avgdl,Ejecución,Posición,Score Total,alice (f=383),rabbit (f=43),queen (f=65),king (f=60),alice (f=3),rabbit (f=2),queen (f=11),king (f=3),alice (f=440),rabbit (f=0),queen (f=172),king (f=62),alice (f=31),rabbit (f=3),queen (f=1036),king (f=3004),alice (f=62),rabbit (f=1),queen (f=1),king (f=5)
0,libro_11 (Alice's Adventures in Wonder),0.28,"Base (k1=1.2, b=0.75)",1,18.7026,5.9639 (tf=383),5.7792 (tf=43),3.7375 (tf=65),3.2219 (tf=60),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,libro_11 (Alice's Adventures in Wonder),0.28,"Sin ajuste por longitud (k1=1.2, b=0.0)",1,18.5376,5.9539 (tf=383),5.6947 (tf=43),3.7010 (tf=65),3.1879 (tf=60),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,libro_11 (Alice's Adventures in Wonder),0.28,"Mayor efecto de las repeticiones (k1=2.0, b=0.75)",1,25.3743,8.1248 (tf=383),7.8145 (tf=43),5.0681 (tf=65),4.3669 (tf=60),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,libro_45 (Anne of Green Gables),1.07,"Base (k1=1.2, b=0.75)",2,13.4619,NaN,NaN,NaN,NaN,4.2036 (tf=3),3.5884 (tf=2),3.3813 (tf=11),2.2886 (tf=3),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,libro_45 (Anne of Green Gables),1.07,"Sin ajuste por longitud (k1=1.2, b=0.0)",3,13.6459,NaN,NaN,NaN,NaN,4.2661 (tf=3),3.6585 (tf=2),3.3986 (tf=11),2.3226 (tf=3),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,libro_45 (Anne of Green Gables),1.07,"Mayor efecto de las repeticiones (k1=2.0, b=0.75)",4,15.5977,NaN,NaN,NaN,NaN,4.7869 (tf=3),3.8898 (tf=2),4.3147 (tf=11),2.6062 (tf=3),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,libro_12 (Through the Looking-Glass),0.31,"Base (k1=1.2, b=0.75)",3,12.9432,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.9647 (tf=440),0.0000 (tf=0),3.7568 (tf=172),3.2217 (tf=62),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,libro_12 (Through the Looking-Glass),0.31,"Sin ajuste por longitud (k1=1.2, b=0.0)",4,12.8895,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.9563 (tf=440),0.0000 (tf=0),3.7433 (tf=172),3.1899 (tf=62),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,libro_12 (Through the Looking-Glass),0.31,"Mayor efecto de las repeticiones (k1=2.0, b=0.75)",2,17.6044,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.1266 (tf=440),0.0000 (tf=0),5.1115 (tf=172),4.3663 (tf=62),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,libro_100 (The Complete Works of Willia),10.74,"Base (k1=1.2, b=0.75)",4,12.8487,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.5197 (tf=31),1.3545 (tf=3),3.7335 (tf=1036),3.2409 (tf=3004),NaN,NaN,NaN,NaN


### Interpretación del efecto de $b$ y $k_1$

1. **Efecto de eliminar el ajuste por longitud ($k_1 = 1.2, b = 0.00$ frente a Base $k_1 = 1.2, b = 0.75$):**
   * El parámetro $b$ controla cuánto penaliza BM25 a los documentos largos (y cuánto bonifica a los cortos) mediante el factor $1 - b + b\frac{|d|}{\text{avgdl}}$ en el denominador.
   * Al fijar $b = 0.00$, ese factor vale $1$ para todos los documentos sin importar su tamaño:
     * **Documentos muy largos suben:** `libro_100` (*The Complete Works of William Shakespeare*, con $|d| = 720\,393$ tokens, es decir, $10.74$ veces la media) sube de la **posición 4** ($12.8487$) a la **posición 2** ($16.9465$). En la configuración base, sus apariciones moderadas de `rabbit` ($f=3$) y `alice` ($f=31$) estaban fuertemente penalizadas por su enorme longitud (aportaban $1.3545$ y $4.5197$); sin ajuste por longitud ($b=0$), sus aportes suben a $4.1811$ y $5.7500$.
     * **Documentos cortos pierden su bonificación:** `libro_11` ($|d|/\text{avgdl} = 0.28$) baja su puntuación de $18.7026$ a $18.5376$, y `libro_12` ($|d|/\text{avgdl} = 0.31$) baja de $12.9432$ a $12.8895$, cayendo de la **posición 3 a la 4**.

2. **Efecto de dar mayor peso a las repeticiones ($k_1 = 2.0, b = 0.75$ frente a Base $k_1 = 1.2, b = 0.75$):**
   * El parámetro $k_1$ controla la velocidad de **saturación** de la frecuencia $f(t,d)$ y el techo máximo de cada término, que tiende a $\text{IDF}(t)\cdot(k_1 + 1)$.
   * Al subir $k_1$ de $1.2$ a $2.0$, los documentos que repiten muchas veces los términos de consulta aumentan mucho más su puntuación que los documentos con pocas apariciones:
     * `libro_12` (*Through the Looking-Glass*), que tiene frecuencias muy altas en tres términos (`alice`=440, `queen`=172, `king`=62), incrementa su score en **$+4.6612$ puntos** ($12.9432 \to 17.6044$) y sube de la **posición 3 a la posición 2**.
     * En cambio, `libro_45` (*Anne of Green Gables*) contiene los cuatro términos pero con frecuencias muy bajas (`alice`=3, `rabbit`=2, `queen`=11, `king`=3); al aumentar $k_1$, el denominador $f(t,d) + k_1(1-b+b|d|/\text{avgdl})$ también crece y frena la ganancia de los términos poco repetidos, por lo que `libro_45` solo gana **$+2.1358$ puntos** ($13.4619 \to 15.5977$) y cae de la **posición 2 a la posición 4** (superado tanto por `libro_12` como por `libro_100`).

3. **Por qué `libro_11` no cambia de posición (se mantiene en #1 en las tres configuraciones):**
   * Aunque la puntuación de `libro_11` (*Alice's Adventures in Wonderland*) cambia en las tres ejecuciones ($18.7026$ en Base, $18.5376$ con $b=0.00$, y $25.3743$ con $k_1=2.0$), permanece en la **posición 1** porque reúne simultáneamente todas las condiciones favorables: es un documento corto ($|d|/\text{avgdl} = 0.28$), contiene **los cuatro términos** de la consulta (incluyendo los más raros `alice` y `rabbit`), y presenta frecuencias altas en todos ellos ($f = 383, 43, 65, 60$). Por ello, ni al quitar la penalización por longitud ($b=0$) ni al premiar más las repeticiones ($k_1=2.0$) ningún otro documento logra alcanzar la suma de sus cuatro aportes.